In [1]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-contrib-python-headless

import cv2
print(cv2.__version__)
print(hasattr(cv2.optflow, "DualTVL1OpticalFlow_create"))

4.13.0
True


In [2]:
import os, sys, shutil

os.chdir("/content")

# Force-clean any previous broken clones (fixes duplicate/path conflicts)
if os.path.exists("/content/Practical-RIFE"):
    shutil.rmtree("/content/Practical-RIFE")

!git clone https://github.com/hzwer/Practical-RIFE.git /content/Practical-RIFE

os.chdir("/content/Practical-RIFE")

# Create weights dir and download flownet.pkl
os.makedirs("/content/Practical-RIFE/train_log", exist_ok=True)
!wget -q -O /content/Practical-RIFE/train_log/flownet.pkl \
    "https://huggingface.co/alibaba-pai/pai-easyphoto/resolve/main/flownet.pkl"

# Install deps
!pip install -q opencv-python-headless torch torchvision numpy pillow

# ---- CRITICAL FIX: force sys.path to include repo root ONLY ONCE ----
repo_path = "/content/Practical-RIFE"
sys.path = [p for p in sys.path if "Practical-RIFE" not in p]  # dedupe
sys.path.insert(0, repo_path)

print("CWD:", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("flownet.pkl exists:", os.path.exists("/content/Practical-RIFE/train_log/flownet.pkl"))

Cloning into '/content/Practical-RIFE'...
remote: Enumerating objects: 633, done.
remote: Counting objects: 100% (393/393), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 633 (delta 378), reused 252 (delta 252), pack-reused 240 (from 1)
Receiving objects: 100% (633/633), 3.04 MiB | 19.71 MiB/s, done.
Resolving deltas: 100% (389/389), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.6 MB/s eta 0:00:00
CWD: /content/Practical-RIFE
sys.path[0]: /content/Practical-RIFE
flownet.pkl exists: True


In [3]:
import cv2
import numpy as np

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read {path} — did you upload it?")
    return img

def compute_tvl1_flow(img0_gray, img1_gray):
    tvl1 = cv2.optflow.DualTVL1OpticalFlow_create()
    flow = tvl1.calc(img0_gray, img1_gray, None)  # shape (H, W, 2)
    return flow

# Paths (uploaded via sidebar into /content/)
frame0_path = "/content/frame_00.png"
frame1_path = "/content/frame_20.png"

img0_gray = load_gray(frame0_path)
img1_gray = load_gray(frame1_path)

flow_0to1 = compute_tvl1_flow(img0_gray, img1_gray)  # F_{0->1}
flow_1to0 = compute_tvl1_flow(img1_gray, img0_gray)  # F_{1->0}

# Scale to t = 0.5
flow_0to_t = 0.5 * flow_0to1
flow_1to_t = 0.5 * flow_1to0

print("Flow shapes:", flow_0to_t.shape, flow_1to_t.shape)

Flow shapes: (1500, 2500, 2) (1500, 2500, 2)


In [5]:

import os
for root, dirs, files in os.walk("/content/Practical-RIFE"):
    if "train_log" in root or "model" in root:
        for f in files:
            if f.endswith(".py"):
                print(os.path.join(root, f))

/content/Practical-RIFE/model/warplayer.py
/content/Practical-RIFE/model/loss.py
/content/Practical-RIFE/model/pytorch_msssim/__init__.py


In [6]:
# FULL DIAGNOSTIC — no filter, saara .py file dikhao
import os
for root, dirs, files in os.walk("/content/Practical-RIFE"):
    if ".git" in root:
        continue
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

/content/Practical-RIFE/inference_video_enhance.py
/content/Practical-RIFE/inference_img.py
/content/Practical-RIFE/inference_video.py
/content/Practical-RIFE/inference_img_SR.py
/content/Practical-RIFE/model/warplayer.py
/content/Practical-RIFE/model/loss.py
/content/Practical-RIFE/model/pytorch_msssim/__init__.py


In [7]:
with open("/content/Practical-RIFE/inference_img.py") as f:
    content = f.read()
print(content[:1500])

import os
import cv2
import torch
import argparse
from torch.nn import functional as F
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True

parser = argparse.ArgumentParser(description='Interpolation for a pair of images')
parser.add_argument('--img', dest='img', nargs=2, required=True)
parser.add_argument('--exp', default=4, type=int)
parser.add_argument('--ratio', default=0, type=float, help='inference ratio between two images with 0 - 1 range')
parser.add_argument('--rthreshold', default=0.02, type=float, help='returns image when actual ratio falls in given range threshold')
parser.add_argument('--rmaxcycles', default=8, type=int, help='limit max number of bisectional cycles')
parser.add_argument('--model', dest='modelDir', type=str, default='train_log', help='directory with 

In [8]:
os.chdir("/content/Practical-RIFE")
if "/content/Practical-RIFE" not in sys.path:
    sys.path.insert(0, "/content/Practical-RIFE")

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("model") or mod_name.startswith("train_log"):
        del sys.modules[mod_name]

# Same cascade jo inference_img.py use karta hai
try:
    try:
        from model.RIFE_HDv2 import Model
        rife_model = Model()
        rife_model.load_model("train_log", -1)
        print("Loaded v2.x HD model.")
    except Exception:
        from train_log.RIFE_HDv3 import Model
        rife_model = Model()
        rife_model.load_model("train_log", -1)
        print("Loaded v3.x HD model.")
except Exception:
    from model.RIFE_HD import Model
    rife_model = Model()
    rife_model.load_model("train_log", -1)
    print("Loaded v1.x HD model.")

rife_model.eval()
rife_model.device()
print("RIFE model loaded successfully.")

ModuleNotFoundError: No module named 'model.RIFE_HD'

In [9]:
# FULL CONTENT DUMP — model/ aur train_log/ folders
import os

print("=== model/ ===")
for f in sorted(os.listdir("/content/Practical-RIFE/model")):
    print(f)

print("\n=== train_log/ ===")
if os.path.exists("/content/Practical-RIFE/train_log"):
    for f in sorted(os.listdir("/content/Practical-RIFE/train_log")):
        print(f)
else:
    print("train_log folder does not exist!")

=== model/ ===
loss.py
pytorch_msssim
warplayer.py

=== train_log/ ===
flownet.pkl


In [10]:
import os, shutil

train_log_dir = "/content/Practical-RIFE/train_log"
os.makedirs(train_log_dir, exist_ok=True)

# 1) Copy warplayer.py and loss.py from model/ into train_log/
#    (relative imports inside RIFE_HDv3.py need them in the same package)
shutil.copy("/content/Practical-RIFE/model/warplayer.py", os.path.join(train_log_dir, "warplayer.py"))
shutil.copy("/content/Practical-RIFE/model/loss.py", os.path.join(train_log_dir, "loss.py"))

# 2) Create __init__.py so train_log/ is a proper package
with open(os.path.join(train_log_dir, "__init__.py"), "w") as f:
    f.write("")

# 3) Write IFNet_HDv3.py
ifnet_hdv3_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from .warplayer import warp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def conv(in_planes, out_planes, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                  padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_planes),
    )


class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c // 2, 3, 2, 1),
            conv(c // 2, c, 3, 2, 1),
        )
        self.convblock0 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock1 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock2 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock3 = nn.Sequential(conv(c, c), conv(c, c))
        self.conv1 = nn.Sequential(
            nn.ConvTranspose2d(c, c // 2, 4, 2, 1),
            nn.PReLU(c // 2),
            nn.ConvTranspose2d(c // 2, 4, 4, 2, 1),
        )
        self.conv2 = nn.Sequential(
            nn.ConvTranspose2d(c, c // 2, 4, 2, 1),
            nn.PReLU(c // 2),
            nn.ConvTranspose2d(c // 2, 1, 4, 2, 1),
        )

    def forward(self, x, flow, scale=1):
        x = F.interpolate(x, scale_factor=1.0 / scale, mode="bilinear",
                           align_corners=False, recompute_scale_factor=False)
        flow = F.interpolate(flow, scale_factor=1.0 / scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False) * 1.0 / scale
        feat = self.conv0(torch.cat((x, flow), 1))
        feat = self.convblock0(feat) + feat
        feat = self.convblock1(feat) + feat
        feat = self.convblock2(feat) + feat
        feat = self.convblock3(feat) + feat
        flow = self.conv1(feat)
        mask = self.conv2(feat)
        flow = F.interpolate(flow, scale_factor=scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False) * scale
        mask = F.interpolate(mask, scale_factor=scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False)
        return flow, mask


class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(7 + 4, c=90)
        self.block1 = IFBlock(7 + 4, c=90)
        self.block2 = IFBlock(7 + 4, c=90)
        self.block_tea = IFBlock(10 + 4, c=90)

    def forward(self, x, scale_list=[4, 2, 1], training=False):
        if training == False:
            channel = x.shape[1] // 2
            img0 = x[:, :channel]
            img1 = x[:, channel:]
            flow_list = []
            merged = []
            mask_list = []
            warped_img0 = img0
            warped_img1 = img1
            flow = (x[:, :4]).detach() * 0
            mask = (x[:, :1]).detach() * 0
            loss_cons = 0
            block = [self.block0, self.block1, self.block2]
            for i in range(3):
                f0, m0 = block[i](torch.cat((warped_img0[:, :3], warped_img1[:, :3], mask), 1), flow, scale=scale_list[i])
                f1, m1 = block[i](
                    torch.cat((warped_img1[:, :3], warped_img0[:, :3], -mask), 1),
                    torch.cat((flow[:, 2:4], flow[:, :2]), 1),
                    scale=scale_list[i],
                )
                flow = flow + (f0 + torch.cat((f1[:, 2:4], f1[:, :2]), 1)) / 2
                mask = mask + (m0 + (-m1)) / 2
                mask_list.append(mask)
                flow_list.append(flow)
                warped_img0 = warp(img0, flow[:, :2])
                warped_img1 = warp(img1, flow[:, 2:4])
                merged.append((warped_img0, warped_img1))
            for i in range(3):
                mask_list[i] = torch.sigmoid(mask_list[i])
                merged[i] = merged[i][0] * mask_list[i] + merged[i][1] * (1 - mask_list[i])
            return flow_list, mask_list[2], merged
'''

with open(os.path.join(train_log_dir, "IFNet_HDv3.py"), "w") as f:
    f.write(ifnet_hdv3_code)

# 4) Write RIFE_HDv3.py
rife_hdv3_code = '''
import torch
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
import torch.optim as optim
import itertools
from .warplayer import warp
from torch.nn.parallel import DistributedDataParallel as DDP
from .IFNet_HDv3 import *
import torch.nn.functional as F
from .loss import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Model:
    def __init__(self, local_rank=-1):
        self.flownet = IFNet()
        self.device()
        self.optimG = AdamW(self.flownet.parameters(), lr=1e-6, weight_decay=1e-4)
        self.epe = EPE()
        self.sobel = SOBEL()
        if local_rank != -1:
            self.flownet = DDP(self.flownet, device_ids=[local_rank], output_device=local_rank)

    def train(self):
        self.flownet.train()

    def eval(self):
        self.flownet.eval()

    def device(self):
        self.flownet.to(device)

    def load_model(self, path, rank=0):
        def convert(param):
            if rank == -1:
                return {k.replace("module.", ""): v for k, v in param.items() if "module." in k}
            else:
                return param

        if rank <= 0:
            if torch.cuda.is_available():
                self.flownet.load_state_dict(convert(torch.load("{}/flownet.pkl".format(path))))
            else:
                self.flownet.load_state_dict(convert(torch.load("{}/flownet.pkl".format(path), map_location="cpu")))

    def save_model(self, path, rank=0):
        if rank == 0:
            torch.save(self.flownet.state_dict(), "{}/flownet.pkl".format(path))

    def inference(self, img0, img1, scale=1.0):
        imgs = torch.cat((img0, img1), 1)
        scale_list = [4 / scale, 2 / scale, 1 / scale]
        flow, mask, merged = self.flownet(imgs, scale_list)
        return merged[2]
'''

with open(os.path.join(train_log_dir, "RIFE_HDv3.py"), "w") as f:
    f.write(rife_hdv3_code)

print("Created files in train_log/:")
for f in sorted(os.listdir(train_log_dir)):
    print(" -", f)

Created files in train_log/:
 - IFNet_HDv3.py
 - RIFE_HDv3.py
 - __init__.py
 - flownet.pkl
 - loss.py
 - warplayer.py


In [12]:
import os, sys, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# PART A: Warping (unchanged)
# ------------------------------------------------------------
def to_tensor_rgb(path):
    img = Image.open(path).convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)  # (1,3,H,W)
    return t.to(device)

def warp_with_flow(img_tensor, flow):
    B, C, H, W = img_tensor.shape
    flow_t = torch.from_numpy(flow).float().to(device)  # (H,W,2)

    yy, xx = torch.meshgrid(
        torch.arange(H, device=device),
        torch.arange(W, device=device),
        indexing="ij"
    )
    grid_x = xx.float() + flow_t[..., 0]
    grid_y = yy.float() + flow_t[..., 1]

    grid_x = 2.0 * grid_x / max(W - 1, 1) - 1.0
    grid_y = 2.0 * grid_y / max(H - 1, 1) - 1.0
    grid = torch.stack((grid_x, grid_y), dim=-1).unsqueeze(0)  # (1,H,W,2)

    warped = F.grid_sample(img_tensor, grid, mode="bilinear",
                            padding_mode="border", align_corners=True)
    return warped

img0_t = to_tensor_rgb(frame0_path)
img1_t = to_tensor_rgb(frame1_path)

W0 = warp_with_flow(img0_t, flow_0to_t)
W1 = warp_with_flow(img1_t, flow_1to_t)

print("Warped shapes:", W0.shape, W1.shape)

# ------------------------------------------------------------
# PART B: Recreate missing RIFE model files
# (RIFE_HDv3.py / IFNet_HDv3.py are NOT in the git repo — they
# normally ship inside the downloaded model package, not the repo)
# ------------------------------------------------------------
train_log_dir = "/content/Practical-RIFE/train_log"
os.makedirs(train_log_dir, exist_ok=True)

shutil.copy("/content/Practical-RIFE/model/warplayer.py", os.path.join(train_log_dir, "warplayer.py"))
shutil.copy("/content/Practical-RIFE/model/loss.py", os.path.join(train_log_dir, "loss.py"))

with open(os.path.join(train_log_dir, "__init__.py"), "w") as f:
    f.write("")

ifnet_hdv3_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from .warplayer import warp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def conv(in_planes, out_planes, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                  padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_planes),
    )


class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c // 2, 3, 2, 1),
            conv(c // 2, c, 3, 2, 1),
        )
        self.convblock0 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock1 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock2 = nn.Sequential(conv(c, c), conv(c, c))
        self.convblock3 = nn.Sequential(conv(c, c), conv(c, c))
        self.conv1 = nn.Sequential(
            nn.ConvTranspose2d(c, c // 2, 4, 2, 1),
            nn.PReLU(c // 2),
            nn.ConvTranspose2d(c // 2, 4, 4, 2, 1),
        )
        self.conv2 = nn.Sequential(
            nn.ConvTranspose2d(c, c // 2, 4, 2, 1),
            nn.PReLU(c // 2),
            nn.ConvTranspose2d(c // 2, 1, 4, 2, 1),
        )

    def forward(self, x, flow, scale=1):
        x = F.interpolate(x, scale_factor=1.0 / scale, mode="bilinear",
                           align_corners=False, recompute_scale_factor=False)
        flow = F.interpolate(flow, scale_factor=1.0 / scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False) * 1.0 / scale
        feat = self.conv0(torch.cat((x, flow), 1))
        feat = self.convblock0(feat) + feat
        feat = self.convblock1(feat) + feat
        feat = self.convblock2(feat) + feat
        feat = self.convblock3(feat) + feat
        flow = self.conv1(feat)
        mask = self.conv2(feat)
        flow = F.interpolate(flow, scale_factor=scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False) * scale
        mask = F.interpolate(mask, scale_factor=scale, mode="bilinear",
                              align_corners=False, recompute_scale_factor=False)
        return flow, mask


class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(7 + 4, c=90)
        self.block1 = IFBlock(7 + 4, c=90)
        self.block2 = IFBlock(7 + 4, c=90)
        self.block_tea = IFBlock(10 + 4, c=90)

    def forward(self, x, scale_list=[4, 2, 1], training=False):
        if training == False:
            channel = x.shape[1] // 2
            img0 = x[:, :channel]
            img1 = x[:, channel:]
            flow_list = []
            merged = []
            mask_list = []
            warped_img0 = img0
            warped_img1 = img1
            flow = (x[:, :4]).detach() * 0
            mask = (x[:, :1]).detach() * 0
            block = [self.block0, self.block1, self.block2]
            for i in range(3):
                f0, m0 = block[i](torch.cat((warped_img0[:, :3], warped_img1[:, :3], mask), 1), flow, scale=scale_list[i])
                f1, m1 = block[i](
                    torch.cat((warped_img1[:, :3], warped_img0[:, :3], -mask), 1),
                    torch.cat((flow[:, 2:4], flow[:, :2]), 1),
                    scale=scale_list[i],
                )
                flow = flow + (f0 + torch.cat((f1[:, 2:4], f1[:, :2]), 1)) / 2
                mask = mask + (m0 + (-m1)) / 2
                mask_list.append(mask)
                flow_list.append(flow)
                warped_img0 = warp(img0, flow[:, :2])
                warped_img1 = warp(img1, flow[:, 2:4])
                merged.append((warped_img0, warped_img1))
            for i in range(3):
                mask_list[i] = torch.sigmoid(mask_list[i])
                merged[i] = merged[i][0] * mask_list[i] + merged[i][1] * (1 - mask_list[i])
            return flow_list, mask_list[2], merged
'''

with open(os.path.join(train_log_dir, "IFNet_HDv3.py"), "w") as f:
    f.write(ifnet_hdv3_code)

rife_hdv3_code = '''
import torch
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
from .warplayer import warp
from torch.nn.parallel import DistributedDataParallel as DDP
from .IFNet_HDv3 import *
import torch.nn.functional as F
from .loss import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Model:
    def __init__(self, local_rank=-1):
        self.flownet = IFNet()
        self.device()
        self.optimG = AdamW(self.flownet.parameters(), lr=1e-6, weight_decay=1e-4)
        self.epe = EPE()
        self.sobel = SOBEL()
        if local_rank != -1:
            self.flownet = DDP(self.flownet, device_ids=[local_rank], output_device=local_rank)

    def train(self):
        self.flownet.train()

    def eval(self):
        self.flownet.eval()

    def device(self):
        self.flownet.to(device)

    def load_model(self, path, rank=0):
        def convert(param):
            if rank == -1:
                return {k.replace("module.", ""): v for k, v in param.items() if "module." in k}
            else:
                return param

        if rank <= 0:
            if torch.cuda.is_available():
                self.flownet.load_state_dict(convert(torch.load("{}/flownet.pkl".format(path))))
            else:
                self.flownet.load_state_dict(convert(torch.load("{}/flownet.pkl".format(path), map_location="cpu")))

    def save_model(self, path, rank=0):
        if rank == 0:
            torch.save(self.flownet.state_dict(), "{}/flownet.pkl".format(path))

    def inference(self, img0, img1, scale=1.0):
        imgs = torch.cat((img0, img1), 1)
        scale_list = [4 / scale, 2 / scale, 1 / scale]
        flow, mask, merged = self.flownet(imgs, scale_list)
        return merged[2]
'''

with open(os.path.join(train_log_dir, "RIFE_HDv3.py"), "w") as f:
    f.write(rife_hdv3_code)

print("Created files in train_log/:")
for fname in sorted(os.listdir(train_log_dir)):
    print(" -", fname)

# ------------------------------------------------------------
# PART C: Load the model
# ------------------------------------------------------------
os.chdir("/content/Practical-RIFE")
if "/content/Practical-RIFE" not in sys.path:
    sys.path.insert(0, "/content/Practical-RIFE")

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("model") or mod_name.startswith("train_log"):
        del sys.modules[mod_name]

from train_log.RIFE_HDv3 import Model

rife_model = Model()
rife_model.load_model("train_log", -1)
rife_model.eval()
rife_model.device()

print("RIFE model loaded successfully.")

Warped shapes: torch.Size([1, 3, 1500, 2500]) torch.Size([1, 3, 1500, 2500])
Created files in train_log/:
 - IFNet_HDv3.py
 - RIFE_HDv3.py
 - __init__.py
 - flownet.pkl
 - loss.py
 - warplayer.py
RIFE model loaded successfully.


In [13]:
import torch.nn.functional as F

def pad_to_multiple(t, multiple=32):
    _, _, h, w = t.shape
    ph = ((h + multiple - 1) // multiple) * multiple
    pw = ((w + multiple - 1) // multiple) * multiple
    padding = (0, pw - w, 0, ph - h)
    return F.pad(t, padding), (h, w)

W0_pad, orig_size = pad_to_multiple(W0)
W1_pad, _ = pad_to_multiple(W1)

with torch.no_grad():
    output = rife_model.inference(W0_pad, W1_pad)  # RIFE forward pass

# Crop back to original size
h, w = orig_size
output = output[:, :, :h, :w]

out_np = (output[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
out_img = Image.fromarray(out_np)

save_path = "/content/frame_10_interpolated.png"
out_img.save(save_path)

print(f"Saved interpolated frame to: {save_path}")

Saved interpolated frame to: /content/frame_10_interpolated.png
